# Modelo de Predicción Para Tasa de Mortalidad Estandarizada Rurales vs Urbanas

In [88]:
!pip install pyreadstat


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [89]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

In [90]:
data_path = 'C:/Users/henry/Documents/jbook/Cacervix/datos/'
data = pd.read_excel(data_path + 'data_final_corregido.xlsx')
# rural_data_cluster = pd.read_excel(data_path + "rural_data_cluster_general_k3.xlsx")
# urbana_data_cluster = pd.read_excel(data_path + "urban_data_cluster_general_k2.xlsx")

In [91]:
data['AREA DE RESIDENCIA'].unique()

array(['Cabecera municipal',
       'Centro poblado (Inspección, corregimiento o caserío)',
       'Rural disperso'], dtype=object)

In [92]:
rural_raw = data[data["AREA DE RESIDENCIA"] != "Cabecera municipal"]
urbana_raw = data[data["AREA DE RESIDENCIA"] == "Cabecera municipal"]

In [93]:
data[['AÑO', 'DEPARTAMENTO', 'Total Mujeres']]

,AÑO,DEPARTAMENTO,Total Mujeres
0,1985,META,192905
1,1985,BOYACÁ,546762
2,1985,META,192905
3,1985,HUILA,345484
4,1985,HUILA,345484
...,...,...,...
58930,2023,RISARALDA,508237
58931,2023,AMAZONAS,41524
58932,2023,META,562262
58933,2023,VALLE DEL CAUCA,2436310


In [94]:
def calcular_tasas(df_zona, nombre_zona):
    resumen = df_zona.groupby(["AÑO", "DEPARTAMENTO DE RESIDENCIA"]).agg(
        DEFUNCIONES=("AÑO", "count"),
        TOTAL_MUJERES=("Total Mujeres", "first")  # tomar solo una por grupo
    ).reset_index()

    resumen["TASA_MORTALIDAD"] = (resumen["DEFUNCIONES"] / resumen["TOTAL_MUJERES"]) * 100000
    resumen["ZONA"] = nombre_zona
    return resumen

In [95]:
tasas_rural = calcular_tasas(rural_raw, "RURAL")
tasas_urbana = calcular_tasas(urbana_raw, "URBANA")
tasas_mortalidad = pd.concat([tasas_rural, tasas_urbana], ignore_index=True)
tasas_unicas = tasas_mortalidad.drop_duplicates(subset=["AÑO", "DEPARTAMENTO DE RESIDENCIA"])


In [96]:
data = data.merge(
    tasas_unicas[["AÑO", "DEPARTAMENTO DE RESIDENCIA", "TASA_MORTALIDAD"]],
    on=["AÑO", "DEPARTAMENTO DE RESIDENCIA"],
    how="left"
)

In [97]:
rural_data = data[data["AREA DE RESIDENCIA"] != "Cabecera municipal"].copy()
urbana_data = data[data["AREA DE RESIDENCIA"] == "Cabecera municipal"].copy()


In [98]:
rural_data.tail(33)

,DEPARTAMENTO,DEPARTAMENTO DE RESIDENCIA,MUNICIPIO,MUNICIPIO DE RESIDENCIA,AÑO,MES,SEX,ESTADO_CIVIL,NIVEL_EDUCACION,GRUPO_ETARIO,...,C_ANT3_DESCRIPCION,C_ANT2_DESCRIPCION,C_ANT1_DESCRIPCION,C_DIR1_DESCRIPCION,AÑO.1,PIB,Total Mujeres,TIENE_COMORBILIDAD,NUM_COMORBILIDADES,TASA_MORTALIDAD
58773,SUCRE,SUCRE,SINCELEJO,SINCELEJO,2023,10,Femenino,Soltero,Primaria,55-59,...,No patología,No patología,No patología,No patología,2023,12705.0,495605,False,0,2.219509
58775,NARIÑO,NARIÑO,PASTO,PASTO,2023,10,Femenino,"Unión Libre, divorciado, otro",Primaria,80-84,...,No patología,No patología,No patología,No patología,2023,23455.0,871313,False,0,2.754464
58780,CÓRDOBA,CÓRDOBA,MONTERÍA,MONTERÍA,2023,10,Femenino,Viudo,Primaria,65-69,...,No patología,No patología,No patología,No patología,2023,28303.0,955208,False,0,5.339151
58782,"BOGOTÁ, D.C.",BOYACÁ,BOGOTÁ D.C.,BOGOTÁ D.C.,2023,2,Femenino,Soltero,Primaria,45-49,...,No patología,No patología,No patología,No patología,2023,395438.0,4120589,False,0,1.975564
58784,NARIÑO,NARIÑO,PASTO,PASTO,2023,11,Femenino,Soltero,Ninguno,75-79,...,No patología,No patología,No patología,No patología,2023,23455.0,871313,False,0,2.754464
58786,ATLÁNTICO,ATLÁNTICO,BARRANQUILLA,BARRANQUILLA,2023,11,Femenino,Soltero,Primaria,50-54,...,No patología,No patología,No patología,No patología,2023,70862.0,1435050,False,0,0.766524
58787,ARAUCA,ARAUCA,ARAUCA,ARAUCA,2023,11,Femenino,Soltero,Secundaria,65-69,...,No patología,No patología,No patología,No patología,2023,8743.0,156828,False,0,3.825847
58790,PUTUMAYO,PUTUMAYO,MOCOA,MOCOA,2023,11,Femenino,Soltero,Superior,55-59,...,No patología,No patología,No patología,No patología,2023,5834.0,191824,False,0,0.688616
58793,RISARALDA,RISARALDA,PEREIRA,PEREIRA,2023,1,Femenino,Viudo,Primaria,55-59,...,No patología,No patología,No patología,No patología,2023,26404.0,508237,False,0,1.574069
58794,SANTANDER,SANTANDER,BUCARAMANGA,BUCARAMANGA,2023,1,Femenino,"Unión Libre, divorciado, otro",Secundaria,45-49,...,No patología,No patología,No patología,No patología,2023,101446.0,1204067,False,0,0.830519


## Modelo de predicción de la tasa de mortalidad estandarizada por zona (Rural vs Urbana)

In [99]:
rural_data.columns.tolist()

['DEPARTAMENTO',
 'DEPARTAMENTO DE RESIDENCIA',
 'MUNICIPIO',
 'MUNICIPIO DE RESIDENCIA',
 'AÑO',
 'MES',
 'SEX',
 'ESTADO_CIVIL',
 'NIVEL_EDUCACION',
 'GRUPO_ETARIO',
 'FECHA',
 'AREA_DEFUN',
 'SITIO_DEFUN',
 'ETNIA',
 'AREA DE RESIDENCIA',
 'SEGURIDAD SOCIAL',
 'C_PAT1_DESCRIPCION',
 'C_ANT3_DESCRIPCION',
 'C_ANT2_DESCRIPCION',
 'C_ANT1_DESCRIPCION',
 'C_DIR1_DESCRIPCION',
 'AÑO.1',
 'PIB',
 'Total Mujeres',
 'TIENE_COMORBILIDAD',
 'NUM_COMORBILIDADES',
 'TASA_MORTALIDAD']

In [120]:
cat_features = ["MUNICIPIO DE RESIDENCIA", "ESTADO_CIVIL", "GRUPO_ETARIO", "ETNIA", "SEGURIDAD SOCIAL", "TIENE_COMORBILIDAD"]
num_features = ["PIB", "Total Mujeres", "NUM_COMORBILIDADES"]

In [121]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features)
])

#### Definición y Entrenamiento del Modelo

In [122]:
#Modeos base
rf = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)


In [123]:
#Ensemble
ensemble = VotingRegressor(estimators=[
    ("rf", rf),
    ("xgb", xgb_model)
])

In [124]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", ensemble)
])

In [125]:
def entrenar_y_evaluar(df, zona):
    print(f"\n--- ZONA: {zona.upper()} ---")

    df = df.sort_values("AÑO")

    # X e y
    features = cat_features + num_features
    X = df[features]
    y = df["TASA_MORTALIDAD"]

    tscv = TimeSeriesSplit(n_splits=5)

    for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        print(f"Split {i+1} → MAE: {mean_absolute_error(y_test, y_pred):.4f}, R²: {r2_score(y_test, y_pred):.4f}")


In [126]:
rural_data.shape

(9524, 27)

In [127]:
rural_data

,DEPARTAMENTO,DEPARTAMENTO DE RESIDENCIA,MUNICIPIO,MUNICIPIO DE RESIDENCIA,AÑO,MES,SEX,ESTADO_CIVIL,NIVEL_EDUCACION,GRUPO_ETARIO,...,C_ANT3_DESCRIPCION,C_ANT2_DESCRIPCION,C_ANT1_DESCRIPCION,C_DIR1_DESCRIPCION,AÑO.1,PIB,Total Mujeres,TIENE_COMORBILIDAD,NUM_COMORBILIDADES,TASA_MORTALIDAD
2,META,META,RESTREPO,RESTREPO,1985,1,Femenino,Casado,Primaria,55-59,...,No patología,No patología,No patología,No patología,1985,74.126,192905,False,0,1.036780
6,CUNDINAMARCA,CUNDINAMARCA,TOCAIMA,TOCAIMA,1985,1,Femenino,Viudo,Superior,75-79,...,No patología,No patología,No patología,No patología,1985,311.571,1014646,False,0,1.379792
9,HUILA,HUILA,GARZÓN,GARZÓN,1985,1,Femenino,Casado,Primaria,50-54,...,No patología,No patología,No patología,No patología,1985,120.326,345484,False,0,1.736694
23,HUILA,HUILA,SAN AGUSTÍN,SAN AGUSTÍN,1985,1,Femenino,Soltero,Primaria,60-64,...,No patología,No patología,No patología,No patología,1985,120.326,345484,False,0,1.736694
47,CUNDINAMARCA,CUNDINAMARCA,GUACHETÁ,GUACHETÁ,1985,3,Femenino,Casado,Ninguno,65-69,...,No patología,No patología,No patología,No patología,1985,311.571,1014646,False,0,1.379792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58881,CAUCA,CAUCA,POPAYÁN,POPAYÁN,2023,1,Femenino,Soltero,Primaria,65-69,...,No patología,No patología,No patología,No patología,2023,28537.000,788691,False,0,3.676979
58883,ANTIOQUIA,ANTIOQUIA,MEDELLÍN,MEDELLÍN,2023,1,Femenino,Soltero,Primaria,75-79,...,No patología,No patología,No patología,No patología,2023,231122.000,3543877,False,0,0.987619
58912,CÓRDOBA,CÓRDOBA,MONTERÍA,MONTERÍA,2023,11,Femenino,Soltero,Primaria,85 y más,...,No patología,No patología,No patología,No patología,2023,28303.000,955208,False,0,5.339151
58917,CÓRDOBA,CÓRDOBA,MONTERÍA,MONTERÍA,2023,12,Femenino,"Unión Libre, divorciado, otro",Primaria,85 y más,...,No patología,No patología,No patología,No patología,2023,28303.000,955208,False,0,5.339151


In [128]:
entrenar_y_evaluar(rural_data, "rural")



--- ZONA: RURAL ---
Split 1 → MAE: 1.2975, R²: -0.4381
Split 2 → MAE: 1.2442, R²: -0.1959
Split 3 → MAE: 0.9616, R²: -0.2975
Split 4 → MAE: 0.9787, R²: 0.1006
Split 5 → MAE: 0.9343, R²: 0.0646


In [ ]:
entrenar_y_evaluar(urbana_data, "urbana")


In [135]:
cat_features = [
    "DEPARTAMENTO DE RESIDENCIA",
    "ESTADO_CIVIL",
    "GRUPO_ETARIO",
    "ETNIA",
    "SEGURIDAD SOCIAL",
    "TIENE_COMORBILIDAD"
]

num_features = [
    "PIB",
    "Total Mujeres",
    "NUM_COMORBILIDADES"
]


In [136]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features)
])


In [137]:
import xgboost as xgb
from sklearn.pipeline import Pipeline

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])


In [138]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

def entrenar_y_evaluar_xgb(df, zona):
    print(f"\n--- ZONA: {zona.upper()} ---")

    df = df.sort_values("AÑO").copy()
    features = cat_features + num_features

    X = df[features]
    y = df["TASA_MORTALIDAD"]

    tscv = TimeSeriesSplit(n_splits=5)

    for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        print(f"Split {i+1} → MAE: {mean_absolute_error(y_test, y_pred):.4f}, R²: {r2_score(y_test, y_pred):.4f}")


In [139]:
entrenar_y_evaluar_xgb(rural_data, "rural")



--- ZONA: RURAL ---
Split 1 → MAE: 1.1244, R²: 0.0105
Split 2 → MAE: 0.8652, R²: 0.4128
Split 3 → MAE: 0.7282, R²: 0.2287
Split 4 → MAE: 0.8878, R²: 0.2699
Split 5 → MAE: 0.6626, R²: 0.3629


In [140]:
entrenar_y_evaluar_xgb(urbana_data, "urbana")



--- ZONA: URBANA ---
Split 1 → MAE: 0.7923, R²: -0.0869
Split 2 → MAE: 0.8856, R²: 0.0151
Split 3 → MAE: 0.6711, R²: 0.0783
Split 4 → MAE: 1.3595, R²: -0.6401
Split 5 → MAE: 1.9312, R²: -4.5884


Escala logarirmimca

In [146]:
cat_features = [ 
    "DEPARTAMENTO DE RESIDENCIA",
    "ESTADO_CIVIL",
    "GRUPO_ETARIO",
    "ETNIA",
    "SEGURIDAD SOCIAL",
    "TIENE_COMORBILIDAD"
]

num_features = [
    "AÑO",                # 🆕 Año como variable predictora
    "PIB",
    "Total Mujeres",      # corregido: usar el mismo nombre que tu DataFrame
    "NUM_COMORBILIDADES"
]
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features)
])
import xgboost as xgb
from sklearn.pipeline import Pipeline

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])
import xgboost as xgb
from sklearn.pipeline import Pipeline

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def entrenar_y_evaluar_xgb_log(df, zona):
    print(f"\n--- ZONA: {zona.upper()} (log TASA_MORTALIDAD) ---")

    df = df.sort_values("AÑO").copy()
    features = cat_features + num_features

    X = df[features]
    y = np.log1p(df["TASA_MORTALIDAD"])  # ← transformación logarítmica

    tscv = TimeSeriesSplit(n_splits=5)

    for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred_log = pipeline.predict(X_test)
        
        # Volver al espacio original para evaluar
        y_pred = np.expm1(y_pred_log)
        y_test_original = np.expm1(y_test)

        print(f"Split {i+1} → MAE: {mean_absolute_error(y_test_original, y_pred):.4f}, R²: {r2_score(y_test_original, y_pred):.4f}")



In [147]:
entrenar_y_evaluar_xgb(rural_data, "rural")



--- ZONA: RURAL ---
Split 1 → MAE: 1.0643, R²: 0.0989
Split 2 → MAE: 0.8735, R²: 0.3953
Split 3 → MAE: 0.7203, R²: 0.2097
Split 4 → MAE: 0.9094, R²: 0.2062
Split 5 → MAE: 0.6086, R²: 0.5037


In [145]:
entrenar_y_evaluar_xgb(urbana_data, "urbana")



--- ZONA: URBANA ---
Split 1 → MAE: 0.7368, R²: 0.0433
Split 2 → MAE: 0.8965, R²: 0.0666
Split 3 → MAE: 0.6378, R²: 0.1554
Split 4 → MAE: 1.3775, R²: -0.6404
Split 5 → MAE: 2.1024, R²: -5.3751
